# Notebook 02 — ReAct, Chain-of-Thought & Tree-of-Thoughts

**Workshop:** Agentic AI for Scientists — Week 2
**Block:** 3 of 6 (30 min)
**Goal:** Understand the *reasoning* patterns behind agents. We go in the order the ideas were invented:

1. **Chain-of-Thought (CoT)** — make the model think out loud. One prompt, no tools.
2. **ReAct** — interleave that thinking with **actions** (tool calls). We build the loop by hand, then collapse it to LangChain's `create_react_agent`.
3. **Tree-of-Thoughts (ToT)** — don't commit to the first chain of reasoning; branch, evaluate, and search.
4. **ReAct vs the tool-calling agent** — the two ways a model can pick a tool, and who carries the responsibility.

**Why this order matters:** if your first exposure to ReAct is `agent = create_react_agent(...)`, you'll think the framework does the reasoning. It doesn't. The reasoning lives in a prompt — and that prompt is just Chain-of-Thought with two extra keywords.


## 0. Setup

In [ ]:
%pip install -q "google-genai>=1.0" \
    "langchain==0.3.7" "langchain-google-genai==2.0.11" "langchain-community==0.3.5" python-dotenv


In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


In [ ]:
import re
from google import genai
from google.genai import types
from langchain_google_genai import ChatGoogleGenerativeAI

client = genai.Client()                                        # raw SDK, for the hand-built ReAct loop
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # LangChain wrapper, for create_react_agent etc.
MODEL = "gemini-2.5-flash"


def as_text(output) -> str:
    """AgentExecutor output is a plain string for both ReAct and Gemini's
    tool-calling agent. Tiny normaliser, kept for safety."""
    if isinstance(output, str):
        return output
    return "".join(getattr(b, "text", "") or "" for b in output)


print(f"Ready with model: {MODEL}")


---

## 1. Chain-of-Thought — the seed of every reasoning agent

CoT (Wei et al., 2022) is almost embarrassingly simple: ask the model to **show its reasoning before answering**. On multi-step problems this alone lifts accuracy a lot, because the model uses its own intermediate tokens as a scratchpad.

Watch a model answer the same question two ways.


In [ ]:
Q = ("A juggler has 16 balls. Half of the balls are golf balls, and half of the "
     "golf balls are blue. How many blue golf balls are there?")

# (a) Direct — force a bare answer
direct = llm.invoke(Q + " Reply with ONLY the final number.").content
print("DIRECT:", direct)

# (b) Chain-of-Thought — five magic words
cot = llm.invoke(Q + " Let's think step by step.").content
print("\nCHAIN-OF-THOUGHT:\n", cot)


The phrase *"Let's think step by step"* is the entire trick of **zero-shot CoT**. The model spends tokens decomposing `16 -> 8 golf balls -> 4 blue` before answering. No framework, no tools — just a prompt that elicits reasoning.

**ReAct is this idea plus one move:** let some of those reasoning steps reach out and *act* on the world (call a tool), then read the result back. So `ReAct = Reason + Act = CoT interleaved with tool calls`.

---

## 2. The ReAct prompt template — verbatim

ReAct (Yao et al., 2022) is a *prompting pattern*, not an API. Read it carefully — this is the whole mechanism.


In [ ]:
REACT_PROMPT = """Answer the following question as best you can. You have access to these tools:

{tools_description}

Use the following format, EXACTLY:

Question: the input question you must answer
Thought: your reasoning about what to do next
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation loop can repeat several times)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {question}
{scratchpad}"""

print(REACT_PROMPT)


Notice the format names the three moving parts of every agent step: **Thought** (the CoT bit), **Action + Action Input** (the tool call), **Observation** (the result fed back). The model writes this text; *we* parse it and run the tools. That's why it's a pattern, not a framework.

---

## 3. Define three tools

Mock implementations so the notebook runs offline and the trace is reproducible. In production, swap each body for a real API call.


In [ ]:
def web_search(query: str) -> str:
    """Mock web search. Keyword-matched (not exact-phrase) so it answers the model's
    query however it phrases it — keeps the classroom trace clean and reproducible."""
    q = query.lower()
    if "react" in q and any(w in q for w in ("author", "wrote", "who")):
        return "Yao et al., ICLR 2023 (arXiv:2210.03629)."
    if "transformer" in q and any(w in q for w in ("year", "publish", "when", "date")):
        return "'Attention Is All You Need' (the transformer paper) was published in 2017 by Vaswani et al."
    if "anthropic" in q and any(w in q for w in ("found", "start", "when")):
        return "Anthropic was founded in 2021 by Dario and Daniela Amodei."
    return f"No results for: {query}"


def calculator(expression: str) -> str:
    expression = expression.replace("^", "**")   # models often write ^ for "to the power of"
    if not re.fullmatch(r"[\d\s+\-*/().]+", expression):
        return "ERROR: only basic arithmetic allowed"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"ERROR: {exc}"


def fetch_paper_abstract(arxiv_id: str) -> str:
    """Mock paper-abstract lookup. Real version would hit the arXiv API."""
    canned = {
        "2210.03629": "ReAct: a paradigm that interleaves reasoning traces and task-specific "
                      "actions in language models for better synergy between the two.",
        "1706.03762": "Attention Is All You Need: the Transformer, a model architecture relying "
                      "entirely on attention, dispensing with recurrence and convolutions.",
    }
    return canned.get(arxiv_id.strip(), f"No abstract found for arXiv:{arxiv_id}")


TOOLS = {
    "web_search": (web_search, "Search the web. Input: a search query string."),
    "calculator": (calculator, "Do arithmetic. Input: a math expression like '2 + 2'."),
    "fetch_paper_abstract": (fetch_paper_abstract, "Get a paper abstract. Input: an arXiv ID like '2210.03629'."),
}


**In production, the mocks become real tools.** Swap `web_search` for a hosted search API built for agents — like **Tavily** — and the tool *interface* stays identical; only the body changes:

```python
# pip install tavily-python   (needs TAVILY_API_KEY)
from langchain_community.tools.tavily_search import TavilySearchResults
real_search = TavilySearchResults(max_results=3)   # drop-in replacement for web_search
```

Everything below runs the same whether the search is mocked or real — that interchangeability is the point of a clean tool interface.


---

## 4. Hand-built ReAct loop

Parse the model's Thought/Action/Action Input lines, run the action, append the Observation, re-prompt. Stop when we see `Final Answer:`. **This is the agent.**


In [ ]:
def format_tools_for_prompt(tools: dict) -> tuple[str, str]:
    desc = "\n".join(f"- {name}: {d}" for name, (_, d) in tools.items())
    return desc, ", ".join(tools.keys())


def parse_action(text: str):
    """Extract (action_name, action_input) from the model's latest output, or None."""
    m = re.search(
        r"Action:\s*([\w_]+)\s*[\r\n]+Action Input:\s*(.+?)(?=[\r\n]+(?:Observation|Thought|Final Answer)|$)",
        text, re.DOTALL,
    )
    if not m:
        return None
    return m.group(1).strip(), m.group(2).strip().strip('"').strip("'")


def run_react(question: str, max_iterations: int = 6) -> str:
    tools_description, tool_names = format_tools_for_prompt(TOOLS)
    scratchpad = ""

    for step in range(max_iterations):
        prompt = REACT_PROMPT.format(
            tools_description=tools_description, tool_names=tool_names,
            question=question, scratchpad=scratchpad,
        )
        # stop_sequences halts the model right after it writes "Action Input",
        # BEFORE it hallucinates its own Observation. WE supply the observation.
        # thinking_budget=0 stops gemini-2.5-flash from adding hidden reasoning
        # that would otherwise swallow the stop sequence and muddy the trace.
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                max_output_tokens=1024,
                stop_sequences=["\nObservation:"],
                thinking_config=types.ThinkingConfig(thinking_budget=0),
            ),
        )
        text = response.text
        print(f"\n--- STEP {step + 1} ---")
        print(text)

        if "Final Answer:" in text:
            return text.split("Final Answer:", 1)[1].strip()

        parsed = parse_action(text)
        if not parsed:
            return f"PARSE ERROR: could not extract an action from:\n{text}"

        action, action_input = parsed
        tool_fn, _ = TOOLS[action]
        observation = tool_fn(action_input)        # <-- run the tool
        print(f"\nObservation: {observation}")     # <-- feed the result back

        scratchpad += text + f"\nObservation: {observation}\n"

    return "Max iterations reached without final answer."


answer = run_react(
    "Who wrote the ReAct paper, and what year was the transformer paper published? "
    "Then compute the difference in years."
)
print("\n=== FINAL ANSWER ===")
print(answer)


**Read the trace top to bottom.** You'll see the loop turn, one tool at a time:

```
Thought: I need the authors of the ReAct paper first.
Action: web_search
Action Input: react paper authors
Observation: Yao et al., ICLR 2023 ...
Thought: Now the transformer year.
Action: web_search
Action Input: transformer paper year
Observation: ... 2017 ...
Thought: Difference is 2023 - 2017.
Action: calculator
Action Input: 2023 - 2017
Observation: 6
Thought: I now know the final answer
Final Answer: ...
```

That structure — **Thought -> Action -> Observation, repeat** — *is* ReAct. The `stop_sequences=["\nObservation:"]` trick is the one subtlety: we stop the model the instant it asks for a tool, so it can't make up the tool's output. The real output is whatever our Python returns.

---

## 5. Same thing, via LangChain `create_react_agent` — traced in LangSmith

Now collapse the loop: same trace shape, but with parsing, retries, and `handle_parsing_errors` provided. And this time we'll **watch the reasoning in [LangSmith](https://smith.langchain.com)** as it runs.

**First, turn on tracing.** LangSmith records every LangChain step — each Thought, Action, tool call, latency, token count — as a nested trace, with no change to your agent code. Classic LangChain 0.3.x reads the **`LANGCHAIN_*`** env names (the website's newer `LANGSMITH_*` names are ignored on this version). Add `LANGCHAIN_API_KEY` as a Colab Secret (free key: smith.langchain.com -> Settings -> API Keys), then run the cell below. No key? It just stays off and everything still runs.


In [ ]:
# Optional LangSmith tracing. Safe to run with no key — tracing just stays off.
import os

def _secret(*names):
    """Return the first of these names found in Colab Secrets or the environment."""
    for n in names:
        try:
            from google.colab import userdata  # type: ignore
            v = userdata.get(n)
            if v:
                return v
        except Exception:
            pass
        if os.environ.get(n):
            return os.environ[n]
    return None

# Colab Secrets don't auto-load into os.environ — read them here and set the names
# classic LangChain expects. Works whether you named them LANGCHAIN_* or LANGSMITH_*.
ls_key = _secret("LANGCHAIN_API_KEY", "LANGSMITH_API_KEY")
if ls_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"           # the switch (NOT LANGSMITH_TRACING on 0.3.x)
    os.environ["LANGCHAIN_API_KEY"]    = ls_key
    os.environ["LANGCHAIN_PROJECT"]    = _secret("LANGCHAIN_PROJECT", "LANGSMITH_PROJECT") or "agentic-ai-week2"
    endpoint = _secret("LANGCHAIN_ENDPOINT", "LANGSMITH_ENDPOINT")   # only needed for the EU region
    if endpoint:
        os.environ["LANGCHAIN_ENDPOINT"] = endpoint
    print("LangSmith tracing ON")
    print("  key loaded : " + ls_key[:6] + "..." + ls_key[-4:] + "  (" + str(len(ls_key)) + " chars)")
    print("  project    : '" + os.environ["LANGCHAIN_PROJECT"] + "'   <- look for THIS name in LangSmith")
    print("  endpoint   : " + os.environ.get("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com (US default)"))
    print("Now run the smoke-test cell below to confirm a trace actually lands.")
else:
    print("No LangSmith key found -> tracing OFF (everything still runs).")
    print("Fix: add a Colab Secret named LANGCHAIN_API_KEY (key icon, left sidebar),")
    print("     and toggle 'Notebook access' ON for it. Then re-run this cell -> it must print 'tracing ON'.")


**Smoke test (run this before the agent).** Colab's tracer posts on a background thread, so if you glance at LangSmith too soon you see nothing. This cell fires one tiny LangChain call and **force-flushes** it — a definitive yes/no in ~2 seconds. If the project doesn't even appear after this, the key isn't loading (re-check the cell above printed `tracing ON`).


In [ ]:
# LangSmith smoke test — one traced call, flushed immediately.
import os
if os.environ.get("LANGCHAIN_TRACING_V2") == "true":
    from langchain_core.tracers.langchain import wait_for_all_tracers
    _ = llm.invoke("Reply with the single word: ok")   # a real LangChain call -> a trace
    wait_for_all_tracers()                              # force the background post NOW (no lag)
    proj = os.environ.get("LANGCHAIN_PROJECT", "default")
    print("Sent 1 test trace -> project '" + proj + "'.")
    print("Open https://smith.langchain.com -> Tracing -> project '" + proj + "'.")
    print("A run named 'ChatGoogleGenerativeAI' there = tracing works. Now run the agent below.")
else:
    print("Tracing is OFF -> fix the key in the cell above first, then re-run this.")


Now run the ReAct agent — if tracing is on, this exact run shows up in your LangSmith project:


In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

# ReAct passes the "Action Input" string straight to the tool, so single-string
# Tool(func=...) wrappers are the natural fit here.
lc_tools = [Tool(name=n, func=fn, description=d) for n, (fn, d) in TOOLS.items()]

# LangChain's stock ReAct prompt needs exactly these vars: tools, tool_names, input, agent_scratchpad.
LC_REACT_PROMPT = PromptTemplate.from_template("""Answer the following question as best you can. You have access to these tools:

{tools}

Use this format:

Question: the input question
Thought: reasoning
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

agent = create_react_agent(llm, lc_tools, LC_REACT_PROMPT)
executor = AgentExecutor(agent=agent, tools=lc_tools, verbose=True,
                         max_iterations=6, handle_parsing_errors=True)

result = executor.invoke({
    "input": "Who wrote the ReAct paper, and what year was the transformer paper published? "
             "Then compute the difference in years.",
})
print("\n=== FINAL ANSWER ===")
print(result["output"])

# Force this run to post to LangSmith now (otherwise it flushes a few seconds later).
try:
    from langchain_core.tracers.langchain import wait_for_all_tracers
    wait_for_all_tracers()
except Exception:
    pass


Same trace, same answer — but now the loop is one line. Because we built it ourselves first, the abstraction is transparent: when LangChain raises `OutputParserException: Could not parse LLM output`, you know it's the regex on `Action:` that failed, and `handle_parsing_errors=True` is what feeds the model a "fix your format" nudge instead of crashing.

**Now open [smith.langchain.com](https://smith.langchain.com).** If you turned tracing on above, this exact `AgentExecutor` run is waiting there as a **trace tree** — expand it to see every Thought → Action → Observation, which tool was called with what input, the latency, and the token counts. That's how you debug a real agent: you *watch the trace* instead of guessing which step went wrong.

> Only **LangChain** calls are traced. The hand-built ReAct loop in section 4 talks to the Gemini SDK directly, so it won't appear in LangSmith — but `create_react_agent` here (and the tool-calling agent in section 8) will.

---

## 6. Live-add a tool

Extending an agent doesn't touch the loop. Add to the tool list; the prompt re-renders with the new tool next iteration.


In [ ]:
new_tool = Tool(
    name="get_arxiv_abstract",
    func=fetch_paper_abstract,
    description="Fetch the abstract of a paper given its arXiv ID, like '2210.03629'.",
)

lc_tools_ext = lc_tools + [new_tool]
agent2 = create_react_agent(llm, lc_tools_ext, LC_REACT_PROMPT)
executor2 = AgentExecutor(agent=agent2, tools=lc_tools_ext, verbose=True,
                          max_iterations=6, handle_parsing_errors=True)

result = executor2.invoke({
    "input": "Get the abstract of arXiv 2210.03629 and tell me in one sentence what ReAct does.",
})
print("\n=== FINAL ANSWER ===")
print(result["output"])


The agent picked up the new tool with **no prompt edit** — because the tool list *is* the agent's vocabulary, re-rendered into the prompt every step. This is why, in agent-land, *tool design* (clear names + descriptions) matters more than long system prompts.

---

## 7. Tree-of-Thoughts — when one chain of reasoning isn't enough

CoT and ReAct commit to a single line of reasoning. If an early step is wrong, the whole chain is wrong. **Tree-of-Thoughts** (Yao et al., 2023) treats reasoning as **search**: at each step, *propose* several candidate thoughts, *evaluate* how promising each is, then *expand* the best — with backtracking.

We show one level of the search machinery on the "Game of 24" (use the four numbers with `+ - * /` to make 24). The full algorithm recurses this to a solution; we keep it to a single propose -> evaluate step so it stays cheap and readable.


In [ ]:
NUMBERS = [4, 9, 10, 13]

def propose_first_steps(numbers, n=3):
    """Ask the model to PROPOSE several candidate first moves (the 'branching' step)."""
    msg = (f"Game of 24. Numbers: {numbers}. Propose {n} DIFFERENT promising first moves. "
           f"A move combines two of the numbers with + - * or /, leaving a new list of 3 numbers. "
           f"Reply as one move per line in the form:  a OP b = c  (remaining: ...)")
    return llm.invoke(msg).content


def evaluate_states(proposals):
    """Ask the model to EVALUATE each candidate as sure / maybe / impossible (the 'pruning' step)."""
    msg = ("For each proposed first move below, judge whether the remaining three numbers can still "
           "reach 24. Label each: SURE, MAYBE, or IMPOSSIBLE, with a 6-word reason.\n\n" + proposals)
    return llm.invoke(msg).content


print("=== PROPOSE (branch) ===")
proposals = propose_first_steps(NUMBERS)
print(proposals)

print("\n=== EVALUATE (prune) ===")
print(evaluate_states(proposals))


That is the heart of ToT: **propose -> evaluate -> keep the promising branches -> expand**, instead of betting everything on the first thought. Real ToT loops this with a search policy (BFS/DFS + a value heuristic) and backtracks out of dead ends — trading more compute for higher accuracy on problems where a single CoT chain routinely fails (planning, puzzles, proof search).

| Pattern | Shape | When to reach for it |
|---|---|---|
| **CoT** | one straight line of reasoning | most single-pass reasoning tasks |
| **ReAct** | a line of reasoning + tool calls | the model needs facts/computation from outside |
| **ToT** | a tree, searched + pruned | one wrong early step dooms the whole answer |

---

## 8. ReAct vs the tool-calling agent — the two paradigms

We've now built tools two ways: the **tool-calling agent** (Notebook 01) and the **ReAct agent** (this notebook). They solve the same job — *pick a tool, run it, feed the result back* — but the responsibility for **picking the right tool** sits in different places.

Run the same question through both and watch the traces.


In [ ]:
from langchain.agents import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# Typed @tool versions work for BOTH agent styles.
@tool
def search(query: str) -> str:
    """Search the web for a fact. Input: a query string."""
    return web_search(query)

@tool
def calc(expression: str) -> str:
    """Do arithmetic. Input: an expression like '2023 - 2017'."""
    return calculator(expression)

shared_tools = [search, calc]
QUESTION = ("Who wrote the ReAct paper, and what year was the transformer paper published? "
            "Then compute the difference in years.")

# (a) ReAct agent — tool choice comes from the ReAct PROMPT + our regex parse
react_tools = [Tool(name=t.name, func=t.func, description=t.description) for t in shared_tools]
react_agent = create_react_agent(llm, react_tools, LC_REACT_PROMPT)
react_exec = AgentExecutor(agent=react_agent, tools=react_tools, verbose=True,
                           max_iterations=6, handle_parsing_errors=True)
print("########## ReAct agent ##########")
react_out = as_text(react_exec.invoke({"input": QUESTION})["output"])

# (b) Tool-calling agent — tool choice comes from the MODEL's native function-calling
tc_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when they help."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
tc_agent = create_tool_calling_agent(llm, shared_tools, tc_prompt)
tc_exec = AgentExecutor(agent=tc_agent, tools=shared_tools, verbose=True, max_iterations=6)
print("\n########## Tool-calling agent ##########")
tc_out = as_text(tc_exec.invoke({"input": QUESTION})["output"])

print("\n=== ReAct answer ===\n", react_out)
print("\n=== Tool-calling answer ===\n", tc_out)


**Same answer, two very different mechanisms.**

| | ReAct agent | Tool-calling (function-calling) agent |
|---|---|---|
| How a tool is chosen | The **ReAct prompt** turns the LLM into a reasoning engine; it writes `Action: / Action Input:` as **text** | The **model vendor's** native function-calling picks the tool + arguments as a structured function-call part |
| Who owns tool selection | **You** — it lives in a prompt you can read and edit | **The vendor** — fine-tuned into the model, hidden from you |
| Parsing | Regex on `Action:` lines (brittle; needs `handle_parsing_errors`) | Structured blocks — no parsing |
| Control / flexibility | **High** — tweak the prompt, change the format, inspect every Thought | **Lower** — you can't see or alter the selection logic |
| Effort / robustness | More to maintain; can mis-format | Less work; more robust out of the box |
| Best when | You need transparency, custom formats, or a model without tool-calling | You're on a modern tool-calling model and want the loop to just work |

This is the "where do we shift the responsibility?" question from the lecture. ReAct keeps tool selection **in a prompt you control**; the function-calling agent **shifts it to the vendor** — less control, much less headache. Most production agents today use function-calling; ReAct remains the clearest way to *understand* what an agent is, and the fallback when a model has no native tool-calling.

---

## 9. Closing exercises (for after class)

1. **Stopping conditions.** Add a token budget to the hand-built loop (stop if cumulative input tokens > 10k). Where does it go?
2. **CoT vs direct, measured.** Run 10 word-problems with and without "Let's think step by step." Count correct answers. How big is the lift?
3. **ToT, fully.** Extend section 7 into a real depth-2 search: expand the top-2 SURE branches one more level. Does it find a solution to Game of 24 for `[4, 9, 10, 13]`?
4. **Force a parse error.** Edit `LC_REACT_PROMPT` to drop the `Action Input:` line. Watch `handle_parsing_errors=True` recover — then set it to `False` and watch it crash. That's the value of building the primitive first.

---

**Next:** [Notebook 03 — RAG End-to-End](03_rag_pipeline.ipynb)
